In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

In [3]:
import datasets 
dataset = datasets.load_dataset('ucberkeley-dlab/measuring-hate-speech')   
df = dataset['train'].to_pandas()
df.describe()

,comment_id,annotator_id,platform,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,hatespeech,hate_speech_score,infitms,outfitms,annotator_severity,std_err,annotator_infitms,annotator_outfitms,hypothesis,annotator_age
count,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.00000,135556.000000,135556.000000,135556.000000,135556.000000,...,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135451.000000
mean,23530.416138,5567.097812,1.281352,2.954307,2.828875,2.56331,2.278638,2.698575,1.846211,1.052045,...,0.744733,-0.567428,1.034322,1.001052,-0.018817,0.300588,1.007158,1.011841,0.014589,37.910772
std,12387.194125,3230.508937,1.023542,1.231552,1.309548,1.38983,1.370876,0.898500,1.402372,1.345706,...,0.932260,2.380003,0.496867,0.791943,0.487261,0.236380,0.269876,0.675863,0.613006,11.641276
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-8.340000,0.100000,0.070000,-1.820000,0.020000,0.390000,0.280000,-1.578693,18.000000
25%,18148.000000,2719.000000,0.000000,2.000000,2.000000,2.00000,1.000000,2.000000,1.000000,0.000000,...,0.000000,-2.330000,0.710000,0.560000,-0.380000,0.030000,0.810000,0.670000,-0.341008,29.000000
50%,20052.000000,5602.500000,1.000000,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,0.000000,...,0.000000,-0.340000,0.960000,0.830000,-0.020000,0.340000,0.970000,0.850000,0.110405,35.000000
75%,32038.250000,8363.000000,2.000000,4.000000,4.000000,4.00000,3.000000,3.000000,3.000000,2.000000,...,2.000000,1.410000,1.300000,1.220000,0.350000,0.420000,1.170000,1.130000,0.449555,45.000000
max,50070.000000,11142.000000,3.000000,4.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,...,2.000000,6.300000,5.900000,9.000000,1.360000,1.900000,2.010000,9.000000,0.987511,81.000000


In [4]:
judaism = df.loc[df['target_religion_jewish'] == True, ['comment_id', 'text', 'hate_speech_score']].drop_duplicates(subset='comment_id')
len(judaism)

1874

In [5]:
judaism.to_csv('data/ucberkeley-dlab_target_jewish.csv', index=False)
print(judaism['hate_speech_score'].describe())

count    1874.000000
mean       -0.857556
std         2.006435
min        -7.940000
25%        -2.180000
50%        -0.680000
75%         0.517500
max         5.090000
Name: hate_speech_score, dtype: float64


### Pilot Codebook Labeling

In [6]:
from dotenv import load_dotenv
import os
import openai
import ast
import json
import csv
import re
from config import UNIVERSAL, INPUT, I_1, I_2, I_3, I_4, I_5, I_NO_1, N_1, N_2, N_NO_1, J_1, J_2, J_3, J_NO_1
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI(api_key=api_key)

In [21]:
# Sanity Check Test 

# test ids and texts
test_ids = [29933, 39476, 40464, 985, 32861, 32448, 27527, 20045]
test_texts = {row['comment_id']: row['text'] for _, row in judaism[judaism['comment_id'].isin(test_ids)].iterrows()}

# prompt blocks: (block_name, block_content, is_not)
blocks = [
    ("I_1", I_1, False),
    ("I_2", I_2, False),
    ("I_3", I_3, False),
    ("I_4", I_4, False),
    ("I_5", I_5, False),
    ("I_NO_1", I_NO_1, True),
    ("N_1", N_1, False),
    ("N_2", N_2, False),
    ("N_NO_1", N_NO_1, True),
    ("J_1", J_1, False),
    ("J_2", J_2, False),
    ("J_3", J_3, False),
    ("J_NO_1", J_NO_1, True),
]

results = {}  # results[comment_id][block_name] = [(code_id, label), ...]

for comment_id in test_ids:
    results[comment_id] = {}
    text = test_texts[comment_id]

    for block_name, block_content, is_not in blocks:
        is_or_is_not = "is NOT" if is_not else "IS"
        prompt = (
            UNIVERSAL.format(ISorisNOT=is_or_is_not)
            + block_content
            + INPUT.format(id=comment_id, text=text)
        )

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            max_tokens=512,
            temperature=0,
        )

        raw = response.choices[0].message.content.strip()

        # parse JSON
        try:
            data = json.loads(raw)
            parsed = [tuple(pair) for pair in list(data.values())[0]]
            results[comment_id][block_name] = parsed
        except Exception as e:
            results[comment_id][block_name] = {"parse_error": str(e), "raw": raw}

# save json
with open("test_results.json", "w") as f:
    json.dump(results, f, indent=2)

# save csv
with open("test_results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["comment_id", "block", "code_id", "label"])
    for comment_id, blocks_dict in results.items():
        for block_name, labels in blocks_dict.items():
            if isinstance(labels, list):
                for code_id, label in labels:
                    writer.writerow([comment_id, block_name, code_id, label])
            else:
                writer.writerow([comment_id, block_name, "PARSE_ERROR", str(labels)])

print("Done. Results saved to test_results.json and test_results.csv")

Done. Results saved to test_results.json and test_results.csv


In [23]:
import pandas as pd

results_df = pd.read_csv("test_results.csv")
non_n = results_df[results_df['label'] != 'N']

for comment_id, group in non_n.groupby('comment_id'):
    print(f"\n{'='*60}")
    print(f"Comment ID: {comment_id}")
    print(f"Text: {test_texts[comment_id]}")
    print(f"\nNon-N Labels:")
    print(group[['block', 'code_id', 'label']].to_string(index=False))


Comment ID: 985
Text: And the liberal Jews fall for it every time.

Non-N Labels:
block               code_id label
  I_1     D2COLLECTIVEBLAME     C
  I_1          D2STEREOTYPE     C
  I_2          E2STEREOTYPE     E
  N_1             N1BELIEFS     O
  N_1    N1NEGATIVEATTITUDE     O
  N_1      N1HOSTILECONDUCT     O
  N_2          N2CONSPIRACY     A
  N_2        N2CONTROLMEDIA     A
  N_2      N2CONTROLECONOMY     A
  N_2          N2CONTROLGOV     A
  N_2        N2CONTROLOTHER     A
  N_2     N3COLLECTIVEBLAME     A
  N_2    N4HIDDENCONSPIRACY     A
  N_2          N4ISRAELHAND     A
  N_2             N5LOYALTY     A
  N_2        N6IDENTITYDENY     A
  N_2     N7COLLECTIVEGUILT     A
  N_2            N8VIOLENCE     A
  N_2          N9INCITEMENT     A
  N_2  N10SELFDETERMINATION     A
  N_2 N11JEWISHRIGHTSDEFINE     A
  N_2  N11SELFDETERMINATION     A
  N_2       N11JEWISHRIGHTS     A
  N_2     N12DOUBLESTANDARD     A
  J_1           A1ESSENTIAL     E
  J_2          A4DOGWHISTLE     O

In [ ]:
# Token estimate
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

# build one sample prompt
sample_prompt = (
    UNIVERSAL.format(ISorisNOT="IS")
    + I_1
    + INPUT.format(id=29933, text=test_texts[29933])
)

tokens = len(enc.encode(sample_prompt))
print(f"Tokens per prompt: {tokens}")
print(f"Total input tokens (13 blocks × 1874 texts): {tokens * 13 * 1874:,}")

Tokens per prompt: 967
Total input tokens (13 blocks × 1874 texts): 23,558,054


## Full Label

In [ ]:
# CONSTRUCT BATCHES

import json
import os
import re

MODEL = "gpt-4o"
OUTPUT_DIR = "batch_pilot"
os.makedirs(OUTPUT_DIR, exist_ok=True)

blocks = [
    ("I_1", I_1, False),
    ("I_2", I_2, False),
    ("I_3", I_3, False),
    ("I_4", I_4, False),
    ("I_5", I_5, False),
    ("I_NO_1", I_NO_1, True),
    ("N_1", N_1, False),
    ("N_2", N_2, False),
    ("N_NO_1", N_NO_1, True),
    ("J_1", J_1, False),
    ("J_2", J_2, False),
    ("J_3", J_3, False),
    ("J_NO_1", J_NO_1, True),
]

all_ids = judaism['comment_id'].tolist()
all_texts = {row['comment_id']: row['text'] for _, row in judaism.iterrows()}

batch_files = []

for block_name, block_content, is_not in blocks:
    is_or_is_not = "is NOT" if is_not else "IS"
    requests = []
    for comment_id in all_ids:
        text = all_texts[comment_id]
        prompt = (
            UNIVERSAL.format(ISorisNOT=is_or_is_not)
            + block_content
            + INPUT.format(id=comment_id, text=text)
        )
        custom_id = f"{block_name}__{comment_id}"
        requests.append({
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "response_format": {"type": "json_object"},
                "max_tokens": 1024,
                "temperature": 0,
            }
        })

    path = f"{OUTPUT_DIR}/batch_{block_name}.jsonl"
    with open(path, "w") as f:
        for r in requests:
            f.write(json.dumps(r) + "\n")
    batch_files.append((block_name, path))
    print(f"Wrote {len(requests)} requests to {path}")

print(f"\nTotal batch files: {len(batch_files)}")

Wrote 1874 requests to batch_pilot/batch_I_1.jsonl
Wrote 1874 requests to batch_pilot/batch_I_2.jsonl
Wrote 1874 requests to batch_pilot/batch_I_3.jsonl
Wrote 1874 requests to batch_pilot/batch_I_4.jsonl
Wrote 1874 requests to batch_pilot/batch_I_5.jsonl
Wrote 1874 requests to batch_pilot/batch_I_NO_1.jsonl
Wrote 1874 requests to batch_pilot/batch_N_1.jsonl
Wrote 1874 requests to batch_pilot/batch_N_2.jsonl
Wrote 1874 requests to batch_pilot/batch_N_NO_1.jsonl
Wrote 1874 requests to batch_pilot/batch_J_1.jsonl
Wrote 1874 requests to batch_pilot/batch_J_2.jsonl
Wrote 1874 requests to batch_pilot/batch_J_3.jsonl
Wrote 1874 requests to batch_pilot/batch_J_NO_1.jsonl

Total batch files: 13


In [ ]:
# SUBMIT
batch_job_map = {}  # key: block_name -> batch_id

for block_name, path in batch_files:
    with open(path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")
    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    batch_job_map[block_name] = batch.id
    print(f"Submitted {block_name}: {batch.id}")

with open(f"{OUTPUT_DIR}/batch_job_map.json", "w") as f:
    json.dump(batch_job_map, f, indent=2)

Submitted I_1: batch_6a3179d4d1d48190b3460d23497cd225
Submitted I_2: batch_6a3179d842a081908dabc76ed3c26032
Submitted I_3: batch_6a3179dc86fc81909e30cee6e2d23a4c
Submitted I_4: batch_6a3179df35cc81908b0df3dbfda374f2
Submitted I_5: batch_6a3179e237d48190a7287a6e293b9aaa
Submitted I_NO_1: batch_6a3179e5bd548190939bccf0d5951f14
Submitted N_1: batch_6a3179e81b808190a00265b6aae6c5c9
Submitted N_2: batch_6a3179ebd5b4819090b98ba0f60f598b
Submitted N_NO_1: batch_6a3179ee7db8819084551796c4094e80
Submitted J_1: batch_6a3179f2d6e08190abdebc84cd88b66a
Submitted J_2: batch_6a3179f56620819085a025b3c5549683
Submitted J_3: batch_6a3179f9a6788190bbf45d0cc13a3bce
Submitted J_NO_1: batch_6a3179fc6f18819083ec8ec2f6342576


In [ ]:
# CHECK STATUS
with open(f"{OUTPUT_DIR}/batch_job_map.json") as f:
    batch_job_map = json.load(f)

for key, job_id in batch_job_map.items():
    batch = client.batches.retrieve(job_id)
    print(f"{key}: {batch.status} ({batch.request_counts})")

In [ ]:
# STORE
all_results = []

with open(f"{OUTPUT_DIR}/batch_job_map.json") as f:
    batch_job_map = json.load(f)

for block_name, job_id in batch_job_map.items():
    batch = client.batches.retrieve(job_id)
    if batch.status != "completed":
        print(f"{block_name} not yet complete: {batch.status}")
        continue

    output_file = client.files.content(batch.output_file_id)
    lines = output_file.text.strip().split("\n")

    for line in lines:
        record = json.loads(line)
        custom_id = record["custom_id"]
        block_name_parsed, comment_id = custom_id.split("__")

        if record["response"]["status_code"] == 200:
            raw = record["response"]["body"]["choices"][0]["message"]["content"]
            try:
                data = json.loads(raw)
                parsed = [tuple(pair) for pair in list(data.values())[0]]
                for code_id, label in parsed:
                    all_results.append({
                        "block": block_name_parsed,
                        "comment_id": comment_id,
                        "code_id": code_id,
                        "label": label,
                    })
            except Exception as e:
                all_results.append({
                    "block": block_name_parsed,
                    "comment_id": comment_id,
                    "code_id": "PARSE_ERROR",
                    "label": str(e),
                })
        else:
            all_results.append({
                "block": block_name_parsed,
                "comment_id": comment_id,
                "code_id": "API_ERROR",
                "label": str(record["response"]["status_code"]),
            })

results_df = pd.DataFrame(all_results)
results_df.to_csv(f"{OUTPUT_DIR}/full_batch_results.csv", index=False)
results_df.to_json(f"{OUTPUT_DIR}/full_batch_results.json", orient="records", indent=2)

summary = results_df.groupby('block').size().reset_index(name='rows_returned')
print(summary.to_string(index=False))
print(f"\nSaved {len(results_df)} total rows to {OUTPUT_DIR}/full_batch_results.csv and .json")

## Create Features

In [ ]:
# Create embedding features

# PCA?

In [ ]:
# One-hot encode labels

### Predict UC Berkeley Discrete Values Using Label and Semantic Features